# L03 · SCM y jerarquía causal de Pearl

**S3 · Lectura: Cap. 3 · 40–55 min**

## La pregunta

¿Estamos describiendo, interviniendo o imaginando un contrafactual?

## Qué harás

Clasificar preguntas y relacionarlas con un modelo causal.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Preguntas clasificadas y un grafo candidato.

> **Idea clave:** El DAG expresa supuestos; no aparece automáticamente por mirar una tabla.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
import numpy as np, pandas as pd
from scipy import stats
rng = setup()

## 1. Un SCM es un programa (§3.1, Listado 3.1)
Un **modelo causal estructural** es un conjunto de ecuaciones, una por variable, donde cada variable es función de sus causas directas y de un ruido exógeno. El ejemplo del libro: la *motivación* $U$ afecta a quién decide participar ($D$) y al resultado ($Y$):

$$U\sim N(0,1),\quad D=\mathbb 1\{U+\varepsilon_D>0{,}3\},\quad Y=0{,}4\,D+0{,}9\,U+\varepsilon_Y$$

El efecto causal de $D$ sobre $Y$ es **0,4** por construcción. Veamos qué mide una asociación y qué mide una intervención.

In [ ]:
# ✏️ Completa tú — simular
# Pista: do(D=d) no condiciona: sobrescribe la ecuación de D.
def simular(n, rng, do_D=None):
    """Simula el SCM. Si do_D no es None, aplica la intervención do(D = do_D)."""
    U = rng.normal(size=n)
    eD = rng.normal(size=n); eY = rng.normal(size=n)
    D = (U + eD > 0.3).astype(float)
    # TODO: si hay intervención, D deja de depender de U y vale la constante pedida
    Y = 0.4 * D + 0.9 * U + eY
    raise NotImplementedError("aplica la intervención sobre D antes de calcular Y")

In [ ]:
n = 50_000
U, D, Y = simular(n, np.random.default_rng(7))
asoc = Y[D == 1].mean() - Y[D == 0].mean()
_, _, Y1 = simular(n, np.random.default_rng(7), do_D=1)
_, _, Y0 = simular(n, np.random.default_rng(7), do_D=0)
print(f"VER   : E[Y|D=1] − E[Y|D=0]      = {asoc:+.3f}")
print(f"HACER : E[Y|do(D=1)] − E[Y|do(D=0)] = {Y1.mean() - Y0.mean():+.3f}   (efecto causal construido = +0.400)")
lab_check("simular con do", lambda: simular(1000, np.random.default_rng(1), do_D=1)[1].std(), [((), 0.0, 1e-12)])

**Mira el resultado.** Los dos números salen del mismo modelo. ¿Por qué difieren tanto y cuál responde «¿qué pasa si le doy el programa a todos?»?

**Respuesta:**

La asociación mezcla el efecto (0,4) con la **diferencia en motivación** entre quienes eligen participar y quienes no (0,9 × la diferencia de $U$). La intervención corta la flecha $U\to D$, así que compara a las mismas personas con y sin programa. Solo la segunda responde la pregunta de política.



## 2. Abducción – acción – predicción (§3.3)
Un **contrafactual** («¿qué habría pasado con *esta* persona si…?») se calcula en tres pasos:
1. **Abducción:** con lo observado, recuperar el ruido de esa unidad ($\varepsilon_Y=Y-0{,}4D-0{,}9U$).
2. **Acción:** modificar la ecuación de $D$ (`do`).
3. **Predicción:** recalcular $Y$ con el mismo ruido.

In [ ]:
# ✏️ Completa tú — contrafactual
# Pista: Debe salir y + 0.4·(d_nuevo − d).
def contrafactual(u, d, y, d_nuevo):
    """Y que habría tenido esta unidad con D = d_nuevo, manteniendo su ruido."""
    raise NotImplementedError("1) abducción de eY, 2) fija D = d_nuevo, 3) recalcula Y")

In [ ]:
i = np.flatnonzero(D == 0)[0]
cf = contrafactual(U[i], D[i], Y[i], 1.0)
print(f"Unidad {i}: observada con D=0 → Y = {Y[i]:+.3f}; contrafactual con D=1 → Y = {cf:+.3f}; diferencia = {cf - Y[i]:+.3f}")
lab_check("contrafactual", contrafactual, [((0.5, 0.0, 1.0, 1.0), 1.4, 1e-9), ((0.5, 1.0, 1.4, 0.0), 1.0, 1e-9)])

**Mira el resultado.** En este SCM lineal el efecto individual es igual para todos (0,4). ¿Qué cambiaría si $Y=0{,}4\,D\,U+\dots$?

**Respuesta:**

El contrafactual dependería de $U$: **cada unidad tendría su efecto**, y el ATE sería solo un promedio. La abducción seguiría igual de necesaria (recuperar el ruido de la unidad).



## 3. ¿Mi grafo es consistente con los datos? (§3.4)
Un grafo **no se prueba**, pero sí se puede **refutar**: cada $d$-separación del grafo implica una independencia condicional que podemos contrastar. Usamos el escenario lineal-gaussiano con grafo conocido (`10_vars_10_edges`): si el grafo es el verdadero, los pares $d$-separados deben tener correlación parcial ≈ 0.

In [ ]:
sc = load_scenario("synthetic/linear_gaussian/10_vars_10_edges")
sc.card()
cols = sc.covariates
X = sc.train[cols].to_numpy(float)
A = sc.adj   # EVALUATION: grafo verdadero (en la práctica sería el grafo que PROPONES)
print("aristas del grafo:", int((A != 0).sum()))

In [ ]:
# Vista rápida de los datos del escenario
import matplotlib.pyplot as plt
import numpy as np
_fig, _ax = plt.subplots(1, 2, figsize=(9, 3.2))
try:
    _scenario = globals().get("sc") or globals().get("lc")
    if _scenario is None:
        raise NameError("No hay un escenario cargado para el gráfico editorial")
    _df = _scenario.train
    if _scenario.treatment is not None and _scenario.outcome is not None:
        _t = _df[_scenario.treatment].to_numpy(float)
        _y = _df[_scenario.outcome].to_numpy(float)
        if np.unique(_t).size == 2:
            _counts = [int((_t == value).sum()) for value in (0, 1)]
            _ax[0].bar(["control", "tratado"], _counts)
            _ax[0].set_title("Asignación")
            _ax[1].boxplot([_y[_t == 0], _y[_t == 1]], labels=["control", "tratado"])
            _ax[1].set_title("Resultado observado")
        else:
            _ax[0].hist(_t, bins=20)
            _ax[0].set_title(_scenario.treatment)
            _ax[1].scatter(_t, _y, s=8, alpha=.35)
            _ax[1].set_title(f"{_scenario.treatment} frente a {_scenario.outcome}")
    elif _scenario.adj is not None:
        _ax[0].imshow(_scenario.adj, cmap="Greys", interpolation="nearest")
        _ax[0].set_title("Grafo del escenario")
        _ax[1].axis("off")
    else:
        _ax[0].axis("off"); _ax[1].axis("off")
    _fig.suptitle(f"Escenario: {_scenario.name}")
    _fig.tight_layout(); plt.show()
finally:
    plt.close(_fig)

In [ ]:
# ✏️ Completa tú — ci_test
# Pista: r_parcial = −P[0,1]/√(P[0,0]·P[1,1]) con P = inversa de corr(X[:, [i,j,*Z]]).
def ci_test(X, i, j, Z=()):
    """p-valor del test z de Fisher de correlación parcial entre columnas i y j dado Z."""
    raise NotImplementedError("correlación parcial vía inversa de la matriz de correlaciones; z = √(n−|Z|−3)·atanh(r)")

In [ ]:
rng_ = np.random.default_rng(0)
print("p-valor de i⟂j|Z con Z=∅ para un par independiente por construcción:",
      round(ci_test(rng_.normal(size=(2000, 2)), 0, 1), 3))
res = []
d = len(cols)
for i in range(d):
    for j in range(i + 1, d):
        sep = dsep(A, [i], [j], [])
        res.append((i, j, sep, ci_test(X, i, j, ())))
res = pd.DataFrame(res, columns=["i", "j", "d_separados_dado_vacio", "p"])
# consistencia: los pares d-separados NO deben rechazarse (p alto); si hay muchos rechazos, el grafo es inconsistente
sep = res[res.d_separados_dado_vacio]
print(f"{len(sep)} pares d-separados: rechazados al 1 %: {(sep.p < .01).sum()} ({(sep.p < .01).mean():.0%})")
def rechazos(A_):
    s_ = [ci_test(X, i, j, ()) for i in range(d) for j in range(i + 1, d) if dsep(A_, [i], [j], [])]
    return np.mean(np.array(s_) < .01) if s_ else 0.0
if sc.source == "benchmark" and rechazos(A.T) < rechazos(A):
    A = A.T; print("nota: se usó la transpuesta de graph_adjmatrix.csv (convención fila→columna invertida)")
res = pd.DataFrame([(i, j, dsep(A, [i], [j], []), ci_test(X, i, j, ())) for i in range(d) for j in range(i + 1, d)],
                   columns=["i", "j", "d_separados_dado_vacio", "p"])
sep = res[res.d_separados_dado_vacio]
dep = res[~res.d_separados_dado_vacio]
print(f"{len(dep)} pares conectados: rechazados al 1 %: {(dep.p < .01).sum()} ({(dep.p < .01).mean():.0%})")
lab_check("ci_test", lambda: ci_test(rng_.normal(size=(3000, 3)), 0, 1, (2,)) > 0.001, [((), True, 0)])

**Mira el resultado.** Si un par $d$-separado se rechaza con p ≈ 0, ¿qué concluyes? ¿Y si un par conectado *no* se rechaza?

**Respuesta:**

En el primer caso el **grafo es inconsistente con los datos** (falta una arista o hay un confusor omitido). En el segundo no hay contradicción: una dependencia puede ser demasiado débil o cancelarse (infidelidad). Los datos pueden **refutar** un grafo, nunca confirmarlo.



## 4. La escalera aplicada (§3.2)
Clasifica cada pregunta en el nivel de la jerarquía de Pearl: **1** = asociación (ver), **2** = intervención (hacer), **3** = contrafactual (imaginar).

In [ ]:
PREGUNTAS = [
    "¿Qué proporción de quienes toman el curso consigue empleo?",
    "Si el gobierno obliga a todos a tomar el curso, ¿qué pasará con el empleo?",
    "María no tomó el curso y no consiguió empleo. ¿Lo habría conseguido si lo hubiera tomado?",
    "¿Está asociado el tabaquismo con el cáncer de pulmón en esta cohorte?",
    "¿Cuánto bajaría la presión media si prohibimos la sal en las cafeterías?",
    "Dado que este cliente se fue, ¿se habría quedado con el descuento?",
]

In [ ]:
# ✏️ Completa tú — NIVELES
# Pista: ¿Veo, hago o imagino?
NIVELES = [None] * 6      # escribe 1, 2 o 3 para cada pregunta
raise NotImplementedError("clasifica las 6 preguntas")

In [ ]:
import hashlib
_H = [hashlib.sha1(f"{i}:{n}".encode()).hexdigest()[:8] for i, n in enumerate([1, 2, 3, 1, 2, 3])]
aciertos = sum(hashlib.sha1(f"{i}:{n}".encode()).hexdigest()[:8] == h for i, (n, h) in enumerate(zip(NIVELES, _H)))
print(f"Aciertos: {aciertos}/6")
assert aciertos >= 5, "repasa: nivel 1 = observar, 2 = intervenir (do), 3 = misma unidad, mundo distinto"

## 5. Supuesto y bitácora

In [ ]:
fila = bitacora(
    pregunta="¿Qué mide una asociación frente a una intervención en un SCM con un confusor?",
    metodo="Simulación del SCM del Listado 3.1 con do(D)",
    estimando="Efecto de D sobre Y (intervención) frente a la diferencia observada",
    numero_ic=f"asociación = {asoc:+.2f}; efecto bajo do = {Y1.mean() - Y0.mean():+.2f} (verdad 0.40; simulación sin IC)",
    supuesto=supuesto("no observable", "declarado y acotado", "el SCM propuesto (aristas y ausencia de otros confusores) es el mecanismo real"),
    decision="continuar", lab="L03")

---
## Qué queda para el libro
Ejercicios del libro cubiertos: la intervención con confusión (§3, efecto 0 con asociación) y el test de independencia condicional. Quedan: el SCM de Simpson del Cap. 1 y el contraste de `nearc4 ⟂ wage | educ` (el dataset de Card no está en el benchmark).

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*